In [ ]:
import itertools

def variables(term):
    """term: 变元名（字符串），或 ('op', t1, t2) 表示一个二元运算节点。"""
    if isinstance(term, str):
        return {term}
    return variables(term[1]) | variables(term[2])

def evaluate(term, env, table):
    if isinstance(term, str):
        return env[term]
    return table[evaluate(term[1], env, table)][evaluate(term[2], env, table)]

def satisfies(law, n, table):
    """law: (lhs, rhs)；检查该定律在 n 阶原群上是否对所有赋值恒成立。"""
    names = sorted(variables(law[0]) | variables(law[1]))
    for vals in itertools.product(range(n), repeat=len(names)):
        env = dict(zip(names, vals))
        if evaluate(law[0], env, table) != evaluate(law[1], env, table):
            return False, env
    return True, None

def find_counterexample(law_a, law_b, n):
    """搜索满足 law_a 但不满足 law_b 的 n 阶原群；找到即证伪 law_a ⇒ law_b。"""
    for flat in itertools.product(range(n), repeat=n * n):
        table = [list(flat[i * n:(i + 1) * n]) for i in range(n)]
        ok_a, _ = satisfies(law_a, n, table)
        if not ok_a:
            continue
        ok_b, witness = satisfies(law_b, n, table)
        if not ok_b:
            return table, witness        # 反例：A 成立，B 在 witness 赋值下失败
    return None, None